## bronze write helper
 - defines write_to_bronze(), the single reusable write function used by every landing-to-bronze notebook.
 - purpose:
   - writes a dataframe into bronze the bronze lakehouse as a managed delta table
 - load types:
   - incremental - replace and partition by batch_id, so re-running a batch replaces only that batch's rows:
   - full - overwrites the entire table on every run
 - output format: delta, with schema evolution enabled ('mergeSchema')
 - used by: "flights-landing-to-bronze", "airport-landing-to-bronze","carrier-landing-to-bronze" (via "%run")

In [ ]:
from pyspark.sql import functions as F

In [ ]:
def write_to_bronze(
    input_df,
    target_table,
    batch_id,
    load_type="incremental"
):
    final_df = input_df 

    writer = (final_df
        .write
        .format("delta")
        .mode("overwrite")
        .option("mergeSchema", "true"))

    if load_type == "incremental":
        writer = writer.option("replaceWhere", f"batch_id = '{batch_id}'").partitionBy("batch_id")

    writer.saveAsTable(target_table)